# V3 Benchmarks Tables

The combining and rendering notebook for Section 7 of the paper A Subset Indexed Attention Cache for Exact Attention over Many Subsets. It reads the two results files written by V3 Benchmarks 2080 and V3 Benchmarks H100, checks that both machines ran the same library under the same protocol on the same 18 inputs, combines their five datasets into the publication dataset of 36 benchmarks, and generates the nine result tables, the figure and the numerical agreement paragraph used in Section 7.d.

v3 is the reported experiment. It differs from v2, the recorded predecessor, in one library function and in the preparation accounting. The cache's trie on run sampler and full power set rows is generated from the row's run descriptor by the successor rule (`generate_run_trie` in `set_attention_cache.py`), with no subset list, and each method is charged for the preparation it needs. Execution is the unchanged computation measured again, not a number interchangeable with v2. The v2 results stay in `E:\apriori-ml-v2\results` and are not read here.

Every benchmark is compared in two modes, and the tables that report both stack them under one header, an Eager section and then a Replay section, in the original one line per input layout. In the eager mode the cache, launched from Python kernel by kernel, is compared with the fastest timed of the nine eager direct combinations. In the replay mode cache replay, the cache call captured once in a CUDA graph and replayed, is compared with the fastest timed of the six masked attention replays, each masked attention combination captured and replayed the same way. Gathered attention builds its indices on the host inside the call and has no replay.

## Inputs

- `results\section7_benchmarks_2080.pkl.lz4` and `results\h100_v3\section7_benchmarks_H100.pkl.lz4`, one per machine, each holding the five DataFrames `benchmarks`, `method_evaluations`, `execution_calls`, `agreement_checks`, `preparation_measurements` and a metadata dict with the manifest hash, the library file hashes, the machine environment, the tolerances, the timing boundaries and `mock = False`.
- Schema `Frozen Benchmark Plan v3, generated cache preparation and per method preparation charges, 2026-11-08`. 17 evaluations per benchmark: the cache, cache replay, the nine eager direct combinations (the reference `math@1024` among them) and the six masked attention replays. 16 agreement rows per benchmark, every evaluation except the reference. Every timed replay passed replay verification before timing. Four preparation components per benchmark, five repeats each, every row carrying `charged_to`, the methods that pay for it.
- Both files come from the v3 library: `set_attention_cache.py` with the run trie generator, and `set_attention_model.py` d7a7e72a, `set_attention_data.py` fd4c5a48, `set_attention_bench.py` 80b9a7ed unchanged from the v2 commit fc4ec03. The library hashes recorded in the two files must be identical, and cell 1 stops if they are not. The manifest hashes differ by construction, each manifest names its machine in every row, so the inputs are compared row by row instead.
- The software stacks differ by machine: PyTorch 2.5.1 with CUDA 12.1 on the 2080, PyTorch 2.8.0 with CUDA 12.8 on the H100. Both are recorded and named in the captions. The H100 ran in a rented pod on an Intel Xeon Platinum 8480+ with 224 visible CPUs and a CPU quota of 23.8 cores. Eager times include host launch overhead and so depend on that host; replay times mostly reflect GPU work.

## Preparation accounting

Four components are timed per benchmark, each the median of five repeats. Input generation is Q, K, V and W regenerated from the seed on attention rows, nothing on training step rows, the same work for every method, reported once and excluded from the method comparison. Subset materialization is the subset list regenerated from the key, charged to masked attention and gathered attention, and to the cache only on the fixed size run and individual sampler rows, where its trie is built from the list. Cache own preparation is the trie, the level plan and its transfer to the GPU, charged to the cache. Masked attention own preparation is the P × n mask on the CPU, charged to masked attention. Method preparation is the sum of the component medians a method is charged for. Method preparation plus execution adds the eager execution median. Both are sums of medians, not end to end timings.

## Outputs

Written to `results\tables\`. Nine LaTeX tables as `.tex` files, `tab_main_attention`, `tab_main_step`, `tab_boundary`, `tab_control`, `tab_prep_attention`, `tab_prep_step`, `tab_totals`, `tab_replay_setup`, `tab_ceiling` (the last only when a ceiling exclusion exists, and a stale copy is removed when there is none), the paragraph as `paragraph.tex`, the derived tables as `derived.pkl.lz4` (one row per benchmark and mode) and `derived_best_mode.pkl.lz4` (one row per benchmark), and the figure as `fig1_speedup.png`. Each `.tex` file is pasted into the Overleaf section unchanged.

## Cells

- Cell 1. Loads both files, checks each (machine name, `mock = False`, the current schema, 18 unique benchmark rows), checks the pair (identical library hashes, identical protocol metadata including tolerances, repeat counts and timing boundaries, TF32 off on both, the same 18 benchmark input ids with identical input hashes, sizes and roles), checks the dataset shapes (unique evaluation ids, the 17 evaluations by method, one reference, 16 agreement rows per benchmark, 5 completed timed repeats behind every timed evaluation, passed replay verification and recorded setup behind every timed replay, a recorded mask transfer behind every timed masked attention replay, preparation repeat indices exactly 1 to 5, the four components present, one charge per component across its repeats), concatenates the five datasets, and prints the evaluation status counts per machine and method.
- Cell 2. Derived fields per benchmark and mode, the best mode fields, the nine tables, the paragraph and the win count table.
- Cell 3. Figure 1, speedup against budget, eager and replay pairings on both machines.

In [1]:
# Estimated run time: under 10 seconds. Load the two machines' results, verify them against each other, combine the five datasets.
import os, sys
import numpy as np, pandas as pd

ROOT = r"E:\apriori-ml-v3"
RESULT_DIR = os.path.join(ROOT, "results")
TABLE_DIR = os.path.join(RESULT_DIR, "tables")            # the .tex files, the paragraph, the derived tables and the figure land here
MACHINES = ["2080", "H100"]
RESULT_PATHS = {"2080": os.path.join(RESULT_DIR, "section7_benchmarks_2080.pkl.lz4"),
                "H100": os.path.join(RESULT_DIR, "h100_v3", "section7_benchmarks_H100.pkl.lz4")}
EXPECTED_BENCHMARKS_PER_MACHINE = 18
SCHEMA_VERSION = "Frozen Benchmark Plan v3, generated cache preparation and per method preparation charges, 2026-11-08"
PREP_COMPONENTS = ["input generation", "subset materialization", "cache own preparation", "masked attention own preparation"]
EXPECTED_METHODS = {"cache": 1, "cache replay": 1, "masked attention": 6, "gathered attention": 3, "masked attention replay": 6}
EVALUATIONS_PER_BENCHMARK = sum(EXPECTED_METHODS.values())        # 17
AGREEMENT_ROWS_PER_BENCHMARK = EVALUATIONS_PER_BENCHMARK - 1      # every evaluation except the reference
REPLAY_METHODS = ("cache replay", "masked attention replay")
# The H100 pod's host, from lscpu and the cgroup cpu.max read on the pod at run time. Not in the results file, so stated here.
H100_HOST = {"cpu": "Intel Xeon Platinum 8480+", "vcpus": 224, "cpu_quota": 23.8}
DATASETS = ("benchmarks", "method_evaluations", "execution_calls", "agreement_checks", "preparation_measurements")

SRC = os.path.join(ROOT, "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
from apriori.serialization import load_from_pickle
os.makedirs(TABLE_DIR, exist_ok=True)

# The H100 file was written by a newer pandas: StringDtype pickles with two constructor arguments and string arrays
# pickle their state as (dtype, data). Prod's pandas takes one argument and (data, dtype, other). This loader accepts
# both forms and converts every string column to plain object strings, so the datasets are ordinary DataFrames.
import pickle, lz4.frame
import pandas as _pd
from pandas.core.arrays.string_ import StringArray as _StringArray

if not getattr(_pd.StringDtype, "_v2_shim", False):
    _orig_init = _pd.StringDtype.__init__
    def _shim_init(self, storage=None, *args, **kwargs):
        _orig_init(self, storage)
    _pd.StringDtype.__init__ = _shim_init
    _pd.StringDtype._v2_shim = True

class _StringArrayCompat(_StringArray):
    def __setstate__(self, state):
        if isinstance(state, tuple) and len(state) == 2 and isinstance(state[0], _pd.StringDtype):
            state = (state[1], state[0], {})
        super().__setstate__(state)

class _Unpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if name.startswith("StringArray") and module.startswith("pandas"):
            return _StringArrayCompat
        if module.startswith("numpy._core"):
            module = "numpy.core" + module[len("numpy._core"):]
        return super().find_class(module, name)

def load_results(path):
    with lz4.frame.open(path, "rb") as f:
        obj = _Unpickler(f).load()
    for k, v in obj.items():
        if isinstance(v, _pd.DataFrame):
            cols = [c for c in v.columns if isinstance(v[c].dtype, _pd.StringDtype)]
            obj[k] = v.astype({c: object for c in cols}) if cols else v
    return obj

def verification_status(v):
    return v.get("status") if isinstance(v, dict) else None

for m, p in RESULT_PATHS.items():
    assert os.path.isfile(p), f"{m}: results file not found at {p}"
saved = {m: load_results(RESULT_PATHS[m]) for m in MACHINES}

# ----- per machine checks -----
for m, s in saved.items():
    md, env = s["metadata"], s["metadata"]["machine_environment"]
    assert md["machine"] == m, f"{m}: file records machine {md['machine']}"
    assert md["mock"] is False, f"{m}: mock results"
    assert md["schema_version"] == SCHEMA_VERSION, f"{m}: schema {md['schema_version']}, a superseded results file"
    assert len(s["benchmarks"]) == EXPECTED_BENCHMARKS_PER_MACHINE, f"{m}: {len(s['benchmarks'])} benchmarks"
    assert s["benchmarks"].benchmark_id.is_unique, f"{m}: duplicate benchmark rows"
    assert (s["benchmarks"].machine == m).all(), f"{m}: benchmark rows from another machine"
    print(f"{m}: {env['gpu']} | torch {env['torch']}, CUDA {env['cuda']} | CPU threads "
          f"{env.get('torch_cpu_threads', 'not recorded')} of {env.get('visible_cpus', 'not recorded')} visible | "
          f"{len(s['benchmarks'])} benchmarks, {len(s['method_evaluations'])} evaluations, {len(s['agreement_checks'])} agreement checks, "
          f"{md['suite_elapsed_s'] / 3600:.2f} h operational")

# ----- the two machines ran the same library and protocol on the same inputs -----
h2080, h100 = saved["2080"]["metadata"], saved["H100"]["metadata"]
assert h2080["implementation_file_sha256"] == h100["implementation_file_sha256"], "library file hashes differ between machines"
# the manifest hashes differ by construction, each manifest names its machine in every row; the inputs are compared per row below
for f in ("schema_version", "tolerances", "timing_boundaries", "agreement_rule", "budget_rule", "ceiling_ms", "warmups",
          "timed_repeats", "preparation_repeats"):
    assert h2080[f] == h100[f], f"protocol metadata differs between machines: {f}"
for f in ("tf32_matmul", "tf32_cudnn"):
    assert h2080["machine_environment"][f] == h100["machine_environment"][f] == False, f"TF32 not off on both machines: {f}"
# torch and CUDA differ by machine (2.5.1 with CUDA 12.1 on the 2080, 2.8.0 with CUDA 12.8 on the H100); both are recorded in METADATA
print("both machines: same library hashes, protocol metadata, tolerances, repeat counts and timing boundaries, TF32 off; torch",
      h2080["machine_environment"]["torch"], "and", h100["machine_environment"]["torch"])

# dataset shape checks, per machine
for m, s in saved.items():
    ev, ag, pr, ex = s["method_evaluations"], s["agreement_checks"], s["preparation_measurements"], s["execution_calls"]
    assert ev.evaluation_id.is_unique, f"{m}: duplicate evaluation ids"
    cnt = ev.groupby(["benchmark_id", "method"]).size().unstack(fill_value=0)
    assert set(cnt.columns) == set(EXPECTED_METHODS), f"{m}: methods {sorted(cnt.columns)}"
    for meth, k in EXPECTED_METHODS.items():
        assert (cnt[meth] == k).all(), f"{m}: a benchmark without exactly {k} {meth} evaluations"
    assert (ev.groupby("benchmark_id").size() == EVALUATIONS_PER_BENCHMARK).all(), f"{m}: a benchmark without exactly {EVALUATIONS_PER_BENCHMARK} evaluations"
    assert (ev.groupby("benchmark_id").is_reference.sum() == 1).all(), f"{m}: a benchmark without exactly one reference"
    ag_b = ag.evaluation_id.str.split("#").str[0]
    assert (ag_b.value_counts() == AGREEMENT_ROWS_PER_BENCHMARK).all() and ag_b.nunique() == len(s["benchmarks"]), \
        f"{m}: a benchmark without exactly {AGREEMENT_ROWS_PER_BENCHMARK} agreement rows"
    timed_ids = set(ev.loc[ev.status == "timed", "evaluation_id"])
    reps = ex[(ex.call_type == "timed repeat") & (ex.status == "completed")].groupby("evaluation_id").size()
    assert timed_ids <= set(reps.index) and (reps.loc[sorted(timed_ids)] == 5).all(), f"{m}: a timed evaluation without 5 completed timed repeats"
    assert (ev.loc[ev.status == "timed", "timed_repeats_completed"] == 5).all(), f"{m}: timed_repeats_completed is not 5 on a timed row"
    rep = ev[ev.method.isin(REPLAY_METHODS) & (ev.status == "timed")]
    assert (rep.replay_verification.map(verification_status) == "passed").all(), f"{m}: a timed replay without passed replay verification"
    assert pd.to_numeric(rep.replay_setup_ms).notna().all(), f"{m}: a timed replay without its setup time"
    mrep = rep[rep.method == "masked attention replay"]
    assert pd.to_numeric(mrep.replay_mask_transfer_ms).notna().all(), f"{m}: a timed masked attention replay without its mask transfer time"
    idx = pr.groupby(["benchmark_id", "preparation_component"]).repeat_index.apply(lambda s: sorted(s.tolist()) == [1, 2, 3, 4, 5])
    assert idx.all(), f"{m}: preparation repeat indices are not exactly 1 to 5"
    assert set(pr.preparation_component) == set(PREP_COMPONENTS), f"{m}: preparation components {sorted(set(pr.preparation_component))}"
    charge = pr.charged_to.map(lambda v: tuple(v) if isinstance(v, (list, tuple)) else v)
    same = charge.groupby([pr.benchmark_id, pr.preparation_component]).nunique(dropna=False)
    assert (same == 1).all(), f"{m}: charged_to differs across the five repeats of a component"
print(f"both machines: unique evaluation ids, {EVALUATIONS_PER_BENCHMARK} evaluations (cache, cache replay, nine eager direct, six masked "
      f"attention replays) and {AGREEMENT_ROWS_PER_BENCHMARK} agreement rows per benchmark, one reference, 5 completed timed repeats per timed "
      f"evaluation, every timed replay verified with its setup recorded, preparation repeats 1 to 5, one charge per component")

b2080 = saved["2080"]["benchmarks"].set_index("benchmark_input_id")
b100 = saved["H100"]["benchmarks"].set_index("benchmark_input_id")
assert set(b2080.index) == set(b100.index), "the machines ran different benchmark inputs"
for f in ("benchmark_input_sha256", "n", "budget_M", "P", "F", "N", "D", "reuse_ratio", "subset_selection", "benchmark_type", "experimental_role"):
    bad = b2080.index[(b2080[f] != b100.loc[b2080.index, f]).to_numpy()]
    assert len(bad) == 0, f"{f} differs between machines for {list(bad)}"
print("both machines: same 18 inputs with identical input hashes and sizes")

# ----- combined datasets -----
combined = {k: pd.concat([saved[m][k] for m in MACHINES], ignore_index=True) for k in DATASETS}
METADATA = {"mock": False, "ceiling_ms": h100["ceiling_ms"], "schema_version": h100["schema_version"],
            "implementation_commit": h100["implementation_commit"], "implementation_file_sha256": h100["implementation_file_sha256"],
            "machines": {m: saved[m]["metadata"]["machine_environment"] for m in MACHINES}, "h100_host": H100_HOST}
benchmarks, method_evaluations, execution_calls, agreement_checks, preparation_measurements = (combined[k] for k in DATASETS)
print({k: len(v) for k, v in combined.items()})
assert benchmarks.benchmark_id.is_unique, "benchmark ids repeat across machines"
_machine = method_evaluations.benchmark_id.map(benchmarks.set_index("benchmark_id").machine).rename("machine")
method_evaluations.groupby([_machine, "method", "status"]).size().unstack(fill_value=0)

2080: NVIDIA GeForce RTX 2080 SUPER | torch 2.5.1+cu121, CUDA 12.1 | CPU threads not recorded of not recorded visible | 18 benchmarks, 306 evaluations, 288 agreement checks, 1.61 h operational
H100: NVIDIA H100 80GB HBM3 | torch 2.8.0+cu128, CUDA 12.8 | CPU threads not recorded of not recorded visible | 18 benchmarks, 306 evaluations, 288 agreement checks, 1.50 h operational
both machines: same library hashes, protocol metadata, tolerances, repeat counts and timing boundaries, TF32 off; torch 2.5.1+cu121 and 2.8.0+cu128
both machines: unique evaluation ids, 17 evaluations (cache, cache replay, nine eager direct, six masked attention replays) and 16 agreement rows per benchmark, one reference, 5 completed timed repeats per timed evaluation, every timed replay verified with its setup recorded, preparation repeats 1 to 5, one charge per component
both machines: same 18 inputs with identical input hashes and sizes
{'benchmarks': 36, 'method_evaluations': 612, 'execution_calls': 3515, 'agre

C:\Users\jakem\AppData\Local\Temp\ipykernel_18804\4257152063.py:140: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = {k: pd.concat([saved[m][k] for m in MACHINES], ignore_index=True) for k in DATASETS}


status                           ceiling exceeded  out of memory  timed
machine method                                                         
2080    cache                                   0              0     18
        cache replay                            0              8     10
        gathered attention                      2             28     24
        masked attention                        0             22     86
        masked attention replay                 0             32     76
H100    cache                                   0              0     18
        cache replay                            0              0     18
        gathered attention                      0             12     42
        masked attention                        0              2    106
        masked attention replay                 0              4    104

## Cell 2. Derived fields, the nine tables and the paragraph

Derived fields, one row per benchmark and mode, 72 rows, saved as `results\tables\derived.pkl.lz4`. For each benchmark and mode: the cache method's status and median execution time (the cache in eager mode, cache replay in replay mode), the fastest timed direct method of that mode (the comparator: one of the nine eager direct combinations, or one of the six masked attention replays), its label, median, peak allocated and reserved memory, the speedup (comparator median divided by cache median, above 1 means the cache is faster, blank when either side was not timed), the replay setup times and the mask transfer time on replay rows, the number of ceiling exclusions and the smallest excluded agreement call, the number of direct methods out of memory, the four preparation component medians (each the median of exactly five completed repeats or blank), whether the cache's trie was built from the materialized list, the cache's and the comparator's method preparation, both methods' preparation plus execution, and their ratio. On a row where the cache ran over run aligned groups, the cache's method preparation uses the single timing for constructing and transferring all group plans in place of the unchunked own preparation. Every charge is read from the rows' `charged_to` and checked: malformed or unknown assignments stop the cell.

Best mode fields, one row per benchmark, 36 rows, saved as `results\tables\derived_best_mode.pkl.lz4`: the faster timed cache mode and its median, the faster timed direct mode with its label and median, and their speedup.

Tables, each written as one `.tex` file in `results\tables\` and pasted into Overleaf unchanged. Every table keeps the original layout: one line per benchmark input, the 2080 and the H100 side by side, short headers. The tables that report both modes stack them under one header: a centered italic Eager row spanning the table opens the eager results and a centered italic Replay row, under a rule, opens the replay results. A column whose printed value is the same in every row is removed and its value stated once in the caption. A column with any blank cell never folds. A Cache time whose cache ran out of memory unchunked and ran over run aligned groups carries the group count as a superscript in parentheses, explained in the caption. The Key column folds when each section has one winner per machine, with one caption sentence per section and machine.

- `tab_main_attention.tex` and `tab_main_step.tex`, main results, 8 and 4 inputs, each listed in the Eager section and again in the Replay section. Eager compares the cache with the fastest of the nine eager direct combinations. Replay compares cache replay with the fastest of the six masked attention replays. The caption explains both, names both software stacks and the H100 host.
- `tab_boundary.tex`, small record boundary (full power set at n 14 and 16) and fixed size run sampler rows, stacked the same way.
- `tab_control.tex`, limited sharing control, each individual sampler input above the run sampler input at the same n and M, the 2080 and the H100 side by side, stacked the same way.
- `tab_prep_attention.tex` and `tab_prep_step.tex`, preparation and peak allocated memory for the Eager rows of the main tables. Input is synthetic input generation, reported once and excluded from the method columns, omitted on the training step table where it is zero. Cache and Direct are method preparation. A row whose cache ran over run aligned groups carries the group count as a superscript, and its Cache preparation is the single timing for constructing and transferring all group plans.
- `tab_totals.tex`, method preparation plus eager execution for the cache and the fastest eager direct combination on the main comparison rows, and their ratio. Sums of component medians, synthetic input generation excluded, replay setup separate. The Key column folds per machine.
- `tab_replay_setup.tex`, for the Replay rows of the main tables: setup of cache replay and of the comparator masked attention replay (the latter including its one mask transfer), and peak reserved memory of both.
- `tab_ceiling.tex`, one row per benchmark and mode with a ceiling exclusion. The Mode column folds into the caption when every row shares it. Not written when there is none.

The paragraph, `paragraph.tex`, per machine: the eager win counts and the replay win counts, each with its speedup range, out of memory counts and benchmarks with no timed direct method in a mode, then a best mode sentence (the faster timed mode on each side), which `BEST_MODE_SENTENCE = False` leaves out. Then the ceiling exclusion count, the replay verification totals, and the agreement totals (checks passed, failed, not run, finite references). Every number in it is computed from the datasets. The cell ends by displaying the win count table for the eager, replay and best mode pairings, which also covers the fixed size run comparison rows and gives each pairing's smallest and largest speedup.

In [9]:
# Estimated run time: under 10 seconds. The code of Benchmark Tables and Figures v2, code cell 2, revised 2026-11-01 for the
# replay results, then the run on the combined datasets.
# Derived publication fields and the LaTeX tables, from benchmarks, method_evaluations and preparation_measurements.
# Each benchmark is compared in two modes, stacked in each table under one header, an Eager section then a Replay section. Eager: the cache
# against the fastest timed of the nine eager direct combinations. Replay: cache replay against the fastest timed of the six masked
# attention replays. Constant columns fold into the caption. The MOCK marker appears in a caption only when METADATA["mock"] is True.

import numpy as np
import pandas as pd

REPEATS_REQUIRED = 5
MODES = ("eager", "replay")
CACHE_OF = {"eager": "cache", "replay": "cache replay"}
DIRECT_OF = {"eager": ("masked attention", "gathered attention"), "replay": ("masked attention replay",)}
DIRECT_COUNT = {"eager": 9, "replay": 6}
DIRECT_COUNT_WORDS = {"eager": "nine eager direct combinations", "replay": "six masked attention replays"}
CHARGEABLE = {"cache", "masked attention", "gathered attention"}

def num(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return np.nan

def verification_status(v):
    return v.get("status") if isinstance(v, dict) else None

def derive(benchmarks, method_evaluations, preparation_measurements):
    """One row per benchmark and mode. Mode eager pairs the cache with the fastest timed eager direct combination, mode
    replay pairs cache replay with the fastest timed masked attention replay."""
    ev = method_evaluations
    out = []
    for _, b in benchmarks.iterrows():
        e = ev[ev.benchmark_id == b.benchmark_id]
        # preparation medians require exactly REPEATS_REQUIRED completed repeats, as the caption states
        prep = preparation_measurements[preparation_measurements.benchmark_id == b.benchmark_id]
        med, charged = {}, {}
        for comp, g in prep.groupby("preparation_component"):
            done = g[g.status == "completed"]
            med[comp] = done.wall_ms.median() if len(done) == REPEATS_REQUIRED else np.nan
            first = g.charged_to.iloc[0]
            if not isinstance(first, (list, tuple)) or not set(first) <= CHARGEABLE:
                raise ValueError(f"{b.benchmark_id}: malformed charged_to for {comp}: {first!r}")
            charged[comp] = tuple(first)
        assert set(med) == set(PREP_COMPONENTS), f"{b.benchmark_id}: preparation components {sorted(med)}"
        assert charged["cache own preparation"] == ("cache",) and charged["masked attention own preparation"] == ("masked attention",), \
            f"{b.benchmark_id}: own preparation charged to {charged}"
        assert {"masked attention", "gathered attention"} <= set(charged["subset materialization"]), \
            f"{b.benchmark_id}: subset materialization not charged to the direct methods"
        def method_preparation(method):
            """Method preparation: the sum of the component medians charged to the method, synthetic input generation excluded.
            A sum of medians, not an end to end timing. NaN if any charged component is blank."""
            parts = [med[c] for c in PREP_COMPONENTS if method in charged[c] and c != "input generation"]
            if not parts:
                raise ValueError(f"{b.benchmark_id}: no preparation charged to {method}")
            return float(sum(parts)) if all(np.isfinite(p) for p in parts) else np.nan
        for mode in MODES:
            cache = e[e.method == CACHE_OF[mode]]
            direct = e[e.method.isin(DIRECT_OF[mode])]
            timed = direct[direct.status == "timed"]
            c = cache.iloc[0] if len(cache) else None
            cache_status = c.status if c is not None else "not run"
            ok = cache_status == "timed"
            cache_ms = num(c.execution_median_ms) if ok else np.nan
            if len(timed):
                k = timed.loc[timed.execution_median_ms.idxmin()]
                comp_id, comp_ms, comp_lab, comp_method = k.evaluation_id, num(k.execution_median_ms), k.direct_combination, k.method
                comp_mib, comp_res = num(k.peak_gpu_allocated_median_mib), num(k.peak_gpu_reserved_median_mib)
                comp_setup, comp_transfer = num(k.replay_setup_ms), num(k.replay_mask_transfer_ms)
            else:
                comp_id, comp_ms, comp_lab, comp_method = None, np.nan, None, None
                comp_mib = comp_res = comp_setup = comp_transfer = np.nan
            speedup = comp_ms / cache_ms if (np.isfinite(cache_ms) and np.isfinite(comp_ms)) else np.nan
            excl = direct[direct.status == "ceiling exceeded"]
            smallest_excl = excl.agreement_call_wall_ms.min() if len(excl) else np.nan
            if comp_method in ("masked attention", "masked attention replay"):
                direct_prep = method_preparation("masked attention")      # subset list and the CPU mask; a replay moves the mask to the GPU once more
            elif comp_method == "gathered attention":
                direct_prep = method_preparation("gathered attention")    # subset list, indices built inside execution
            else:
                direct_prep = np.nan
            cache_prep = method_preparation("cache")
            if ok and mode == "eager" and num(c.cache_groups) > 1:         # grouped run: the one timed build and transfer of the group plans stands in for own preparation
                cache_prep = cache_prep - med.get("cache own preparation", np.nan) + num(c.cache_group_preparation_ms)
            cache_total, direct_total = cache_prep + cache_ms, direct_prep + comp_ms
            out.append(dict(benchmark_id=b.benchmark_id, mode=mode, cache_method=CACHE_OF[mode], cache_status=cache_status,
                            cache_execution_median_ms=cache_ms,
                            cache_peak_mib=num(c.peak_gpu_allocated_median_mib) if ok else np.nan,
                            cache_peak_reserved_mib=num(c.peak_gpu_reserved_median_mib) if ok else np.nan,
                            cache_replay_setup_ms=num(c.replay_setup_ms) if (ok and mode == "replay") else np.nan,
                            cache_groups=num(c.cache_groups) if (ok and mode == "eager") else np.nan,
                            cache_group_preparation_ms=num(c.cache_group_preparation_ms) if (ok and mode == "eager") else np.nan,
                            cache_replay_verification=verification_status(c.replay_verification) if (c is not None and mode == "replay") else None,
                            comparator_evaluation_id=comp_id, comparator_execution_median_ms=comp_ms,
                            comparator_direct_combination=comp_lab, comparator_method=comp_method,
                            comparator_peak_mib=comp_mib, comparator_peak_reserved_mib=comp_res,
                            comparator_replay_setup_ms=comp_setup if mode == "replay" else np.nan,
                            comparator_mask_transfer_ms=comp_transfer if mode == "replay" else np.nan,
                            speedup=speedup, cache_faster=(speedup > 1) if np.isfinite(speedup) else None,
                            direct_evaluations=int(len(direct)),
                            ceiling_exclusions=int(len(excl)),
                            smallest_excluded_call_ms=smallest_excl,
                            smallest_excluded_call_over_comparator=(smallest_excl / comp_ms if np.isfinite(smallest_excl) and np.isfinite(comp_ms) else np.nan),
                            direct_out_of_memory=int((direct.status == "out of memory").sum()),
                            direct_timed=int(len(timed)),
                            input_generation_median_ms=med.get("input generation", np.nan),
                            subset_materialization_median_ms=med.get("subset materialization", np.nan),
                            cache_own_preparation_median_ms=med.get("cache own preparation", np.nan),
                            masked_own_preparation_median_ms=med.get("masked attention own preparation", np.nan),
                            cache_materializes=("cache" in charged.get("subset materialization", ())),
                            cache_method_preparation_ms=cache_prep,
                            direct_method_preparation_ms=direct_prep,
                            cache_preparation_plus_execution_ms=cache_total,
                            direct_preparation_plus_execution_ms=direct_total,
                            preparation_plus_execution_ratio=(direct_total / cache_total
                                                              if np.isfinite(direct_total) and np.isfinite(cache_total) else np.nan)))
    return benchmarks.merge(pd.DataFrame(out), on="benchmark_id")

def best_modes(benchmarks, derived):
    """One row per benchmark: the faster timed cache mode against the faster timed direct mode."""
    rows = []
    for bid, g in derived.groupby("benchmark_id", sort=False):
        g = g.set_index("mode")
        def fastest(col):
            c = [(g.loc[m, col], m) for m in MODES if np.isfinite(g.loc[m, col])]
            return min(c) if c else (np.nan, None)
        cms, cmode = fastest("cache_execution_median_ms")
        dms, dmode = fastest("comparator_execution_median_ms")
        sp = dms / cms if (np.isfinite(cms) and np.isfinite(dms)) else np.nan
        rows.append(dict(benchmark_id=bid, best_cache_mode=cmode, best_cache_method=CACHE_OF.get(cmode), best_cache_ms=cms,
                         best_direct_mode=dmode, best_direct_combination=g.loc[dmode, "comparator_direct_combination"] if dmode else None,
                         best_direct_ms=dms, best_speedup=sp, best_cache_faster=(sp > 1) if np.isfinite(sp) else None))
    return benchmarks.merge(pd.DataFrame(rows), on="benchmark_id")

def mock_note(metadata):
    return " MOCK VALUES, ESTIMATED FROM V1, NOT MEASURED." if metadata.get("mock", False) else ""

def fmt_ms(v, status=None):
    if status is not None and status != "timed":
        return {"out of memory": "OOM", "ceiling exceeded": "ceil."}.get(status, status)
    return "" if not np.isfinite(v) else (f"{v:,.1f}" if v < 1000 else f"{v:,.0f}")

def fmt_sp(v):
    return "" if not np.isfinite(v) else f"{v:.2f}"

def fmt_int(M):
    return f"{int(M):,}"

def fmt_mib(v, status=None):
    if status is not None and status != "timed":
        return "OOM" if status == "out of memory" else status
    return "" if not np.isfinite(v) else f"{v:,.0f}"

KEY = {"math@all": "a", "math@4096": "b", "math@1024": "c", "default@all": "d", "default@4096": "e", "default@1024": "f",
       "gathered@all": "g", "gathered@4096": "h", "gathered@1024": "i"}
KEY_TEXT = ("Key names the fastest direct combination. Masked attention, mathematical backend, is a unchunked, b at chunk size "
            "$4{,}096$ and c at $1{,}024$. The default backend is d, e and f, and gathered attention is g, h and i, in the same chunk "
            "size order.")
COMBO_WORDS = {"math": "masked attention, mathematical backend", "default": "masked attention, default backend", "gathered": "gathered attention"}

def strip_replay(label):
    return label[:-len(" replay")] if isinstance(label, str) and label.endswith(" replay") else label

def key_of(label):
    return KEY.get(strip_replay(label), "") if label else ""

def combo_words(label):
    head, chunk = strip_replay(label).split("@")
    num_ = f"{int(chunk):,}".replace(",", "{,}") if chunk != "all" else None
    return COMBO_WORDS[head] + (", unchunked" if num_ is None else f", chunk size ${num_}$")

def main_rows(d, mode):
    """Main comparison rows of one mode, in table order, and the row keys from the 2080."""
    m = d[(d.experimental_role == "main comparison") & (d["mode"] == mode)].copy()
    m["type_order"] = (m.benchmark_type != "attention benchmark").astype(int)
    m = m.sort_values(["type_order", "n", "budget_M", "requested_run_length_lambda"])
    keys = [(r.benchmark_type, r.n, r.budget_M, r.requested_run_length_lambda) for _, r in m[m.machine == "2080"].iterrows()]
    return m, keys

def pick(m, machine, bt, n, M, lam):
    return m[(m.machine == machine) & (m.benchmark_type == bt) & (m.n == n) & (m.budget_M == M)
             & (m.requested_run_length_lambda == lam)].iloc[0]

PAIR_FIELDS = ["benchmark_input_id", "benchmark_input_sha256", "n", "budget_M", "P", "reuse_ratio", "subset_selection",
               "benchmark_type", "experimental_role", "mode"]

def check_pair(a, h):
    """The 2080 and H100 rows printed side by side must be the same benchmark input in the same mode. Raises otherwise."""
    for f in PAIR_FIELDS:
        if f in a.index and f in h.index and a[f] != h[f]:
            raise ValueError(f"machine rows differ in {f}: {a.benchmark_id} has {a[f]}, {h.benchmark_id} has {h[f]}")

def paired(m, keys):
    a = [pick(m, "2080", *k) for k in keys]
    h = [pick(m, "H100", *k) for k in keys]
    for x, y in zip(a, h):
        check_pair(x, y)
    return a, h

# A column is a dict: header, align, values (list of printed strings), group (optional spanner title),
# fold (how to describe the column in the caption when its value is the same in every row, or None to never fold).
def col(header, align, values, group=None, fold=None, subgroup=None):
    return dict(header=header, align=align, values=list(values), group=group, fold=fold, subgroup=subgroup)

def fold_constant_columns(cols):
    """Remove columns whose printed value is identical in every row. Returns kept columns and the caption sentence.
    A column with a blank cell never folds, so a missing outcome is never folded into a claim."""
    kept, folded = [], []
    for c in cols:
        vals = set(c["values"])
        if c["fold"] is not None and len(vals) == 1 and len(c["values"]) > 1 and "" not in vals:
            folded.append(c["fold"](next(iter(vals))))
        else:
            kept.append(c)
    if not folded:
        sentence = ""
    elif len(folded) == 1:
        sentence = " In every row, " + folded[0] + "."
    else:
        sentence = " In every row, " + ", ".join(folded[:-1]) + ", and " + folded[-1] + "."
    return kept, sentence

def render(cols, groups_order, row_breaks=()):
    """LaTeX tabular from columns. groups_order lists spanner titles in order; a spanner is printed only if
    at least one of its columns survived folding. Columns with a subgroup get a second spanner row under the
    group row. row_breaks maps a row index to the rule inserted before it (\\midrule or \\addlinespace)."""
    ordered, spec_parts = [], []
    for g in [None] + list(groups_order):
        gc = [c for c in cols if c["group"] == g]
        if gc:
            ordered += gc
            spec_parts.append("".join(c["align"] for c in gc))
    spec = " ".join(spec_parts)

    def spanner_row(key, title):
        """One spanner row. key(c) identifies a run of columns, title(k) is what the run prints, None prints nothing."""
        cells, cmid, i, j = [], [], 1, 0
        while j < len(ordered):
            k = key(ordered[j])
            w = 1
            while k is not None and j + w < len(ordered) and key(ordered[j + w]) == k:
                w += 1
            if k is None:
                cells.append("")
            else:
                cells.append(f"\\multicolumn{{{w}}}{{c}}{{{title(k)}}}")
                cmid.append(f"\\cmidrule(lr){{{i}-{i + w - 1}}}")
            i += w
            j += w
        return (" & ".join(cells) + " \\\\\n" + " ".join(cmid) + "\n") if cmid else ""

    out = f"\\begin{{tabular}}{{{spec}}}\n\\toprule\n"
    out += spanner_row(lambda c: c["group"], lambda k: k)
    if any(c["subgroup"] for c in ordered):
        out += spanner_row(lambda c: (c["group"], c["subgroup"]) if c["subgroup"] else None, lambda k: k[1])
    lines = []
    for r in range(len(ordered[0]["values"])):
        if r in row_breaks:
            lines.append(row_breaks[r])
        lines.append(" & ".join(c["values"][r] for c in ordered) + r" \\")
    out += " & ".join(c["header"] for c in ordered) + " \\\\\n\\midrule\n" + "\n".join(lines) + "\n\\bottomrule\n\\end{tabular}"
    return out

def wrap(caption, label, body, size=r"\footnotesize", colsep="2.5pt"):
    return (f"\\begin{{table}}[tbp]\n\\caption{{{caption}}}\n\\label{{{label}}}\n{size}\n"
            f"\\setlength{{\\tabcolsep}}{{{colsep}}}\n{body}\n\\end{{table}}")

LABEL_OF_KEY = {k: lab for lab, k in KEY.items()}

def host_sentence(metadata):
    """Names the two software stacks and the H100 host, from METADATA."""
    mach = metadata.get("machines", {})
    host = metadata.get("h100_host", {})
    t80, t100 = mach.get("2080", {}).get("torch", "?"), mach.get("H100", {}).get("torch", "?")
    return (f" PyTorch was {t80.split('+')[0]} on the 2080 and {t100.split('+')[0]} on the H100, whose host was a "
            f"{host.get('cpu_quota', 22.1)} core share of an {host.get('cpu', 'Intel Xeon Platinum 8470')}.")

def group_mark(r):
    """A superscript with the group count when the cache ran over run aligned groups, else nothing."""
    g = num(r.get("cache_groups", np.nan))
    return f"$^{{({int(g)})}}$" if np.isfinite(g) and g > 1 else ""

def cache_cell(r):
    return fmt_ms(r.cache_execution_median_ms, r.cache_status) + group_mark(r)

def group_sentence(rows):
    if not any(group_mark(r) for r in rows):
        return ""
    return (" A superscript $(g)$ on a Cache time means the cache ran out of memory unchunked and ran over $g$ groups of the "
            "sorted subsets.")

MAIN_CAPTION = ("Median execution time in milliseconds of the cache and the fastest timed direct method, run sampler, $n = 300$ and "
                "$496$. Each row is one input, identical on both machines. $M$ is the requested budget, $\\lambda$ the requested run "
                "length, $P$ the number of subsets and $\\rho = N / F$ the reuse ratio. Eager rows compare the cache with the fastest of "
                "nine direct combinations launched kernel by kernel. Replay rows compare cache replay with the fastest of six masked "
                "attention replays, each captured once as a CUDA graph. Preparation and replay setup are excluded "
                "(Table~\\ref{tab:replay_setup}). Speedup is Direct divided by Cache. OOM means the cache ran out of memory.")

def _table_main_one(d, metadata, btype, label, title):
    """One table per benchmark type, the eager rows and then the replay rows under one header."""
    a, h, starts = [], [], {}
    for mode in MODES:
        m, keys = main_rows(d, mode)
        keys = [k for k in keys if k[0] == btype]
        x, y = paired(m, keys)
        starts[len(a)] = mode
        a += x
        h += y
    def mcols(rows, machine):
        return [col("Key", "c", [key_of(r.comparator_direct_combination) for r in rows], machine),
                col("Cache", "r", [cache_cell(r) for r in rows], machine),
                col("Direct", "r", [fmt_ms(r.comparator_execution_median_ms) for r in rows], machine),
                col("Speedup", "r", [fmt_sp(r.speedup) for r in rows], machine)]
    cols = [col("$n$", "r", [str(r.n) for r in a]), col("$M$", "r", [fmt_int(r.budget_M) for r in a]),
            col("$\\lambda$", "l", ["$n$" if r.requested_run_length_lambda == r.n else "$10n$" for r in a],
                fold=lambda v: f"the requested run length is $\\lambda = {v.strip(chr(36))}$"),
            col("$P$", "r", [fmt_int(r.P) for r in a]), col("$\\rho$", "r", [f"{r.reuse_ratio:.4f}" for r in a])]
    cols += mcols(a, "2080") + mcols(h, "H100")
    letters = [key_of(r.comparator_direct_combination) for r in a] + [key_of(r.comparator_direct_combination) for r in h]
    groups = [(r["mode"], "2080") for r in a] + [(r["mode"], "H100") for r in h]
    keep_key, key_text = stacked_key(letters, groups)
    if not keep_key:
        cols = [c for c in cols if c["header"] != "Key"]
    kept, folded = fold_constant_columns(cols)
    cap = title + " " + MAIN_CAPTION + folded + key_text + stacked_missing(a + h) + group_sentence(a + h) + host_sentence(metadata)
    return wrap(cap + mock_note(metadata), label, render(kept, ["2080", "H100"], section_breaks(starts, len(kept))))

def table_main_attention(d, metadata):
    return _table_main_one(d, metadata, "attention benchmark", "tab:main_attention", "Main results, attention benchmarks.")

def table_main_step(d, metadata):
    return _table_main_one(d, metadata, "training step benchmark", "tab:main_step", "Main results, training step benchmarks.")

MODE_TITLE = {"eager": "Eager", "replay": "Replay"}

def section_breaks(starts, n_cols, inner=None):
    """Row breaks for a table stacked by mode. starts maps a row index to the mode whose rows begin there; a full width
    centered italic row naming the mode is inserted before it, under a rule when it is not the first section. inner holds the
    breaks within a section."""
    br = dict(inner or {})
    for i, mode in starts.items():
        label = f"\\multicolumn{{{n_cols}}}{{c}}{{\\textit{{{MODE_TITLE[mode]}}}}} \\\\"
        br[i] = label if i == 0 else "\\midrule\n" + label
    return br

def stacked_key(letters, groups):
    """letters: the Key letter of every row. groups: (mode, machine) of every row. When every (mode, machine) group has
    one nonblank letter, returns (False, sentence) and the Key column folds into the caption; otherwise (True, text)."""
    by = {}
    for l, g in zip(letters, groups):
        by.setdefault(g, set()).add(l)
    if all(len(v) == 1 and "" not in v for v in by.values()):
        parts = []
        for machine in ["2080", "H100"]:
            e, r = by.get(("eager", machine)), by.get(("replay", machine))
            if e and r:
                parts.append(f"On the {machine} the fastest direct method was " + combo_words(LABEL_OF_KEY[next(iter(e))])
                             + " in the Eager rows and " + combo_words(LABEL_OF_KEY[next(iter(r))]) + " in the Replay rows.")
            elif e or r:
                v, sec = (e, "Eager") if e else (r, "Replay")
                parts.append(f"On the {machine} the fastest direct method was " + combo_words(LABEL_OF_KEY[next(iter(v))])
                             + f" in the {sec} rows.")
        return False, " " + " ".join(parts)
    return True, " " + KEY_TEXT + " Replay rows use a to f."

STACK_TEXT = ("Eager rows time calls launched kernel by kernel. Replay rows time one replay of a call captured once as a CUDA "
              "graph.")

def stacked_missing(rows):
    return (" A blank Direct means no direct method was timed."
            if any(r.comparator_direct_combination is None for r in rows) else "")

def table_boundary(d, metadata):
    roles = ["small record boundary", "fixed size run comparison"]
    a, h, starts = [], [], {}
    for mode in MODES:
        b = d[d.experimental_role.isin(roles) & (d["mode"] == mode)].copy()
        b["ro"] = b.experimental_role.map({r: i for i, r in enumerate(roles)})
        b = b.sort_values(["ro", "n"])
        starts[len(a)] = mode
        a += [g[g.machine == "2080"].iloc[0] for _, g in b.groupby(["ro", "n"], sort=True)]
        h += [g[g.machine == "H100"].iloc[0] for _, g in b.groupby(["ro", "n"], sort=True)]
    for x, y in zip(a, h):
        check_pair(x, y)
    def mcols(rows, machine):
        return [col("Key", "c", [key_of(r.comparator_direct_combination) for r in rows], machine),
                col("Cache", "r", [cache_cell(r) for r in rows], machine),
                col("Direct", "r", [fmt_ms(r.comparator_execution_median_ms) for r in rows], machine),
                col("Speedup", "r", [fmt_sp(r.speedup) for r in rows], machine)]
    cols = [col("Subset selection", "l", [r.subset_selection for r in a]), col("$n$", "r", [str(r.n) for r in a]),
            col("$M$", "r", [fmt_int(r.budget_M) for r in a]), col("$P$", "r", [fmt_int(r.P) for r in a]),
            col("$\\rho$", "r", [f"{r.reuse_ratio:.4f}" for r in a])]
    cols += mcols(a, "2080") + mcols(h, "H100")
    letters = [key_of(r.comparator_direct_combination) for r in a] + [key_of(r.comparator_direct_combination) for r in h]
    groups = [(r["mode"], "2080") for r in a] + [(r["mode"], "H100") for r in h]
    keep_key, key_text = stacked_key(letters, groups)
    if not keep_key:
        cols = [c for c in cols if c["header"] != "Key"]
    kept, folded = fold_constant_columns(cols)
    cap = ("Boundary and sampler comparison. The full power set rows are the small record boundary at $n = 14$ and $16$. The fixed "
           "size run sampler rows use the sampler of Section~\\ref{sec:counting:sampler}. $M$ is the requested budget, $P$ the number "
           "of subsets and $\\rho = N / F$ the reuse ratio. " + STACK_TEXT + " Times are medians in milliseconds, and Speedup is "
           "Direct divided by Cache. OOM means the cache method ran out of memory.")
    return wrap(cap + folded + key_text + stacked_missing(a + h) + group_sentence(a + h) + mock_note(metadata), "tab:boundary",
                render(kept, ["2080", "H100"], section_breaks(starts, len(kept))))

def table_control(d, metadata):
    """Each control input beside the run sampler input at the same n and M, the 2080 and the H100 side by side, the eager
    rows and then the replay rows under one header."""
    a, h, inner, starts = [], [], {}, {}
    for mode in MODES:
        dm = d[d["mode"] == mode]
        ctl = dm[dm.experimental_role == "limited sharing control"]
        starts[len(a)] = mode
        for k, n in enumerate(sorted(ctl.n.unique())):
            if k > 0:
                inner[len(a)] = r"\addlinespace"
            for machine, out in (("2080", a), ("H100", h)):
                c = ctl[(ctl.machine == machine) & (ctl.n == n)].iloc[0]
                r = dm[(dm.machine == machine) & (dm.n == n) & (dm.budget_M == c.budget_M) & (dm.benchmark_type == "attention benchmark")
                       & (dm.subset_selection == "run sampler") & (dm.requested_run_length_lambda == 10 * n)].iloc[0]
                out += [c, r]
    for x, y in zip(a, h):
        check_pair(x, y)
    def mcols(rows, machine):
        return [col("Key", "c", [key_of(r.comparator_direct_combination) for r in rows], machine),
                col("Cache", "r", [cache_cell(r) for r in rows], machine),
                col("Direct", "r", [fmt_ms(r.comparator_execution_median_ms) for r in rows], machine),
                col("Speedup", "r", [fmt_sp(r.speedup) for r in rows], machine)]
    cols = [col("Subset selection", "l", [r.subset_selection for r in a]), col("$n$", "r", [str(r.n) for r in a]),
            col("$P$", "r", [fmt_int(r.P) for r in a]), col("$\\rho$", "r", [f"{r.reuse_ratio:.4f}" for r in a])]
    cols += mcols(a, "2080") + mcols(h, "H100")
    letters = [key_of(r.comparator_direct_combination) for r in a] + [key_of(r.comparator_direct_combination) for r in h]
    groups = [(r["mode"], "2080") for r in a] + [(r["mode"], "H100") for r in h]
    keep_key, key_text = stacked_key(letters, groups)
    if not keep_key:
        cols = [c for c in cols if c["header"] != "Key"]
    kept, folded = fold_constant_columns(cols)
    cap = ("Limited sharing control, attention benchmarks at $M = 64{,}000$. The individual sampler of "
           "Section~\\ref{sec:counting:sampler} takes single subsets spread evenly through the sorted order, so few prefixes are shared "
           "and the reuse ratio $\\rho = N / F$ is close to $1$. Each run sampler row is the input of Table~\\ref{tab:main_attention} at "
           "the same $n$ and $M$. $P$ is the number of subsets. " + STACK_TEXT + " Times are medians in milliseconds, and Speedup is "
           "Direct divided by Cache. The control also changes the subsets and the trie, so it does not isolate reuse. Control rows are "
           "not counted in any win count.")
    return wrap(cap + folded + key_text + stacked_missing(a + h) + group_sentence(a + h) + mock_note(metadata), "tab:control",
                render(kept, ["2080", "H100"], section_breaks(starts, len(kept), inner)))

def prep_cache_cell(r):
    """The cache's method preparation. On a grouped row the one timed build and transfer of the group plans stands in for the
    unchunked own preparation, marked."""
    return fmt_ms(r.cache_method_preparation_ms) + group_mark(r)

PREP_CAPTION = ("Preparation is in milliseconds, each component the median of five repeats, blank if fewer completed. Input is synthetic "
                "input generation, $Q$, $K$, $V$ and $W$ from the seed, the same work for every method, reported once and excluded from the "
                "method columns. Cache and Direct are method preparation, the sum of the component medians the method needs, not an end to "
                "end timing. On these run sampler rows Cache is trie generation from the run descriptor by the successor rule, level plan "
                "construction and transfer to the GPU; the fixed size run and individual sampler rows of Tables~\\ref{tab:boundary} "
                "and~\\ref{tab:control} regenerate the subset list and build the trie from it. Direct is the subset list regenerated from the key and, for a masked attention "
                "comparator, the $P \\times n$ mask on the CPU. Memory is the median peak GPU memory allocated during execution, in MiB. "
                "OOM means the cache ran out of memory.")

def _table_prep_one(d, metadata, btype, label, title):
    m, keys = main_rows(d, mode="eager")
    keys = [k for k in keys if k[0] == btype]
    a, h = paired(m, keys)
    def mcols(rows, machine):
        inp = ([col("Input", "r", [fmt_ms(r.input_generation_median_ms) for r in rows], machine, subgroup="Preparation, ms")]
               if btype == "attention benchmark" else [])      # training step rows have no synthetic input generation
        return inp + [col("Cache", "r", [prep_cache_cell(r) for r in rows], machine, subgroup="Preparation, ms"),
                col("Direct", "r", [fmt_ms(r.direct_method_preparation_ms) for r in rows], machine, subgroup="Preparation, ms"),
                col("Cache", "r", [fmt_mib(r.cache_peak_mib, r.cache_status) for r in rows], machine, subgroup="Memory, MiB"),
                col("Direct", "r", [fmt_mib(r.comparator_peak_mib) for r in rows], machine, subgroup="Memory, MiB")]
    cols = [col("$n$", "r", [str(r.n) for r in a]), col("$M$", "r", [fmt_int(r.budget_M) for r in a]),
            col("$\\lambda$", "l", ["$n$" if r.requested_run_length_lambda == r.n else "$10n$" for r in a], fold=lambda v: f"the requested run length is $\\lambda = {v.strip(chr(36))}$")]
    cols += mcols(a, "2080") + mcols(h, "H100")
    kept, folded = fold_constant_columns(cols)
    grouped = any(group_mark(r) for r in a + h)
    gtext = (" On a row marked $(g)$ the cache ran over $g$ groups after its unchunked call ran out of memory. Its Cache preparation "
             "is a single timing, not a median of five, the one build of the $g$ group tries from the row's rank intervals with their "
             "plans and transfers; the interval reconstruction itself, microseconds, is outside that clock. Its memory is that of the "
             "grouped run.") if grouped else ""
    return wrap(title + " " + PREP_CAPTION + folded + gtext + mock_note(metadata), label, render(kept, ["2080", "H100"]))

def table_prep_attention(d, metadata):
    return _table_prep_one(d, metadata, "attention benchmark", "tab:prep_attention",
                           "Preparation and memory for the attention benchmarks of Table~\\ref{tab:main_attention}, same rows and order.")

def table_prep_step(d, metadata):
    return _table_prep_one(d, metadata, "training step benchmark", "tab:prep_step",
                           "Preparation and memory for the training step benchmarks of Table~\\ref{tab:main_step}, same rows and order. "
                           "Training step rows have no synthetic input generation, the model and its state come from the file, so the "
                           "Input column is omitted.")

def table_totals(d, metadata):
    """Method preparation plus eager execution, cache against the fastest eager direct combination, main comparison rows."""
    m, keys = main_rows(d, mode="eager")
    a, h = paired(m, keys)
    def mcols(rows, machine):
        ks = sorted({key_of(r.comparator_direct_combination) for r in rows})
        key = [] if len(ks) == 1 else [col("Key", "c", [key_of(r.comparator_direct_combination) for r in rows], machine)]
        return key + [col("Cache", "r", [fmt_ms(r.cache_preparation_plus_execution_ms, r.cache_status) + group_mark(r) for r in rows], machine),
                      col("Direct", "r", [fmt_ms(r.direct_preparation_plus_execution_ms) for r in rows], machine),
                      col("Ratio", "r", [fmt_sp(r.preparation_plus_execution_ratio) for r in rows], machine)]
    def key_sentence(rows, machine):
        ks = sorted({key_of(r.comparator_direct_combination) for r in rows})
        return f" On the {machine} the fastest direct combination is {ks[0]} in every row." if len(ks) == 1 else ""
    cols = [col("Type", "l", ["Attention" if r.benchmark_type.startswith("attention") else "Training step" for r in a]),
            col("$n$", "r", [str(r.n) for r in a]), col("$M$", "r", [fmt_int(r.budget_M) for r in a])]
    cols += mcols(a, "2080") + mcols(h, "H100")
    key_text = " " + KEY_TEXT + key_sentence(a, "2080") + key_sentence(h, "H100")
    kept, folded = fold_constant_columns(cols)
    grouped = any(group_mark(r) for r in a + h)
    cap = ("Method preparation plus eager execution, in milliseconds. Cache is the cache's method preparation of Tables~\\ref{tab:prep_attention} "
           "and~\\ref{tab:prep_step} plus its eager execution median. Direct is the method preparation of the fastest eager direct combination "
           "of Tables~\\ref{tab:main_attention} and~\\ref{tab:main_step}, chosen by execution time, plus that execution median. Each total is a "
           "sum of component medians, not an end to end timing, and synthetic input generation is excluded from both sides. Replay setup is "
           "separate, in Table~\\ref{tab:replay_setup}. Ratio is Direct divided by Cache.")
    gtext = (" On a row marked $(g)$ the cache ran over $g$ groups, and its preparation is the single timing for constructing and "
             "transferring all $g$ group plans, as in Table~\\ref{tab:prep_attention} or~\\ref{tab:prep_step}.") if grouped else ""
    return wrap(cap + folded + key_text + gtext + mock_note(metadata), "tab:totals", render(kept, ["2080", "H100"]))

def table_replay_setup(d, metadata):
    """Replay setup and memory for the rows of the two replay tables, attention rows first."""
    m, keys = main_rows(d, mode="replay")
    a, h = paired(m, keys)
    def direct_setup(r):
        v = r.comparator_replay_setup_ms + r.comparator_mask_transfer_ms
        return fmt_ms(v)
    def mcols(rows, machine):
        return [col("Cache", "r", [fmt_ms(r.cache_replay_setup_ms, r.cache_status) for r in rows], machine, subgroup="Setup, ms"),
                col("Direct", "r", [direct_setup(r) for r in rows], machine, subgroup="Setup, ms"),
                col("Cache", "r", [fmt_mib(r.cache_peak_reserved_mib, r.cache_status) for r in rows], machine, subgroup="Memory, MiB"),
                col("Direct", "r", [fmt_mib(r.comparator_peak_reserved_mib) for r in rows], machine, subgroup="Memory, MiB")]
    cols = [col("Benchmark", "l", ["attention" if r.benchmark_type.startswith("attention") else "training step" for r in a]),
            col("$n$", "r", [str(r.n) for r in a]), col("$M$", "r", [fmt_int(r.budget_M) for r in a]),
            col("$\\lambda$", "l", ["$n$" if r.requested_run_length_lambda == r.n else "$10n$" for r in a],
                fold=lambda v: f"the requested run length is $\\lambda = {v.strip(chr(36))}$")]
    cols += mcols(a, "2080") + mcols(h, "H100")
    kept, folded = fold_constant_columns(cols)
    first_step = next((i for i, r in enumerate(a) if not r.benchmark_type.startswith("attention")), None)
    breaks = {first_step: r"\midrule"} if first_step else {}
    cap = ("Replay setup and memory for the Replay rows of Tables~\\ref{tab:main_attention} and~\\ref{tab:main_step}. Setup is "
           "the one time graph warmup and capture in milliseconds, plus the mask transfer for Direct, all excluded from execution. "
           "Memory is the median peak GPU memory reserved during a replay, in MiB, which includes the graph's pool. It is not comparable "
           "with the allocated memory of Tables~\\ref{tab:prep_attention} and~\\ref{tab:prep_step}. OOM means cache replay ran out of "
           "memory. A blank Direct means no masked attention replay was timed.")
    return wrap(cap + folded + mock_note(metadata), "tab:replay_setup", render(kept, ["2080", "H100"], breaks))

def table_ceiling(d, metadata):
    """One row per benchmark and mode with at least one ceiling exclusion. Returns None when there is none."""
    c = d[d.ceiling_exclusions > 0].copy()
    if len(c) == 0:
        return None
    c["type_order"] = (c.benchmark_type != "attention benchmark").astype(int)
    c["mo"] = c["mode"].map({md: i for i, md in enumerate(MODES)})
    c = c.sort_values(["machine", "type_order", "experimental_role", "n", "budget_M", "requested_run_length_lambda", "mo"])
    rows = [r for _, r in c.iterrows()]
    def lam(r):
        return "" if pd.isna(r.requested_run_length_lambda) else ("$n$" if r.requested_run_length_lambda == r.n else "$10n$")
    cols = [col("Machine", "l", [r.machine for r in rows], fold=lambda v: f"the machine is the {v}"),
            col("Benchmark", "l", ["attention" if r.benchmark_type.startswith("attention") else "training step" for r in rows],
                fold=lambda v: f"the benchmark type is {v}"),
            col("Subset selection", "l", [r.subset_selection for r in rows], fold=lambda v: f"the subset selection is the {v}"),
            col("$n$", "r", [str(r.n) for r in rows], fold=lambda v: f"$n = {v}$"),
            col("$M$", "r", [fmt_int(r.budget_M) for r in rows], fold=lambda v: f"$M = {v.replace(',', '{,}')}$"),
            col("$\\lambda$", "l", [lam(r) for r in rows], fold=lambda v: f"$\\lambda = {v.strip('$')}$"),
            col("Mode", "l", [r["mode"] for r in rows],
                fold=lambda v: "the excluded methods are eager direct combinations" if v == "eager"
                else "the excluded methods are masked attention replays"),
            col("Excluded", "r", [f"{r.ceiling_exclusions} of {r.direct_evaluations}" for r in rows],
                fold=lambda v: f"{v} direct methods exceeded the ceiling"),
            col("Smallest call", "r", [fmt_ms(r.smallest_excluded_call_ms) for r in rows],
                fold=lambda v: f"the smallest excluded agreement call took {v}~ms"),
            col("Comparator", "r", [fmt_ms(r.comparator_execution_median_ms) for r in rows]),
            col("Ratio", "r", [fmt_sp(r.smallest_excluded_call_over_comparator) for r in rows])]
    kept, folded = fold_constant_columns(cols)
    heads = {c["header"] for c in kept}
    cap = ("Ceiling exclusions. Benchmarks in which a direct method's agreement call took longer than the $30$~s ceiling and was "
           "not timed.")
    if "Mode" in heads:
        cap += " Mode says whether the excluded methods are eager direct combinations or masked attention replays."
    if "Excluded" in heads:
        cap += " Excluded counts such methods out of the nine eager or six replay methods of the mode."
    if "Smallest call" in heads:
        cap += " Smallest call is the shortest excluded call in milliseconds."
    cap += (" Comparator is the fastest timed direct median of the same mode in milliseconds. Ratio is the smallest excluded call "
            "divided by Comparator and does not estimate the excluded median.")
    return wrap(cap + folded + mock_note(metadata), "tab:ceiling", render(kept, []), colsep="4pt")

WIN_ROLES = ["main comparison", "fixed size run comparison"]

def win_counts(d, best):
    """Per machine, benchmark type and role: the eager pairing, the replay pairing and the best mode pairing."""
    rows = []
    def add(pairing, keys, g, sp, cache_missing):
        comparable = sp[np.isfinite(sp)]
        rows.append(dict(pairing=pairing, machine=keys[0], benchmark_type=keys[1], experimental_role=keys[2], benchmarks=len(g),
                         comparable=len(comparable), cache_faster=int((comparable > 1).sum()),
                         min_speedup=float(comparable.min()) if len(comparable) else np.nan,
                         max_speedup=float(comparable.max()) if len(comparable) else np.nan,
                         cache_out_of_memory=cache_missing,
                         missing=int(len(g) - len(comparable) - cache_missing)))
        return rows[-1]
    dm = d[d.experimental_role.isin(WIN_ROLES)]
    for mode in MODES:
        for keys, g in dm[dm["mode"] == mode].groupby(["machine", "benchmark_type", "experimental_role"]):
            r = add(mode, keys, g, g.speedup.to_numpy(dtype=float), int((g.cache_status == "out of memory").sum()))
            r.update(direct_out_of_memory=int(g.direct_out_of_memory.sum()), no_direct_timed=int((g.direct_timed == 0).sum()),
                     benchmarks_with_ceiling_exclusions=int((g.ceiling_exclusions > 0).sum()), ceiling_exclusions=int(g.ceiling_exclusions.sum()))
    bm = best[best.experimental_role.isin(WIN_ROLES)]
    for keys, g in bm.groupby(["machine", "benchmark_type", "experimental_role"]):
        add("best mode", keys, g, g.best_speedup.to_numpy(dtype=float), int(g.best_cache_mode.isna().sum()))
    return pd.DataFrame(rows)

def _range(r):
    return "" if not r.comparable else (f", speedup ${r.min_speedup:.2f}$" if r.comparable == 1 else
                                         f", speedups ${r.min_speedup:.2f}$ to ${r.max_speedup:.2f}$")

BEST_MODE_SENTENCE = True     # the closing best mode sentence per machine; set False to leave it out of paragraph.tex

def paragraph(d, best, agreement_checks, method_evaluations):
    """The win count, replay verification and numerical agreement paragraph, every number computed from the datasets."""
    wc = win_counts(d, best)
    wc = wc[wc.experimental_role == "main comparison"]
    parts = []
    for machine in ["H100", "2080"]:
        g = wc[wc.machine == machine]
        def get(pairing, bt):
            return g[(g.pairing == pairing) & (g.benchmark_type == bt)].iloc[0]
        ea, et = get("eager", "attention benchmark"), get("eager", "training step benchmark")
        ra, rt = get("replay", "attention benchmark"), get("replay", "training step benchmark")
        parts.append(f"On the {machine} the cache was faster than the fastest eager direct combination in {ea.cache_faster} of "
                     f"{ea.comparable} attention benchmarks{_range(ea)}, and {et.cache_faster} of {et.comparable} training step "
                     f"benchmarks{_range(et)}.")
        parts.append(f"Cache replay was faster than the fastest masked attention replay in {ra.cache_faster} of {ra.comparable} "
                     f"attention benchmarks{_range(ra)}, and {rt.cache_faster} of {rt.comparable} training step benchmarks{_range(rt)}.")
        for (x, y), who, other in (((ea, et), "The cache", "no eager direct combination"),
                                   ((ra, rt), "Cache replay", "no masked attention replay")):
            oom = [f"{v.cache_out_of_memory} {w}" for v, w in ((x, "attention"), (y, "training step")) if v.cache_out_of_memory]
            if oom:
                parts.append(f"{who} ran out of memory in {' and '.join(oom)} benchmarks on the {machine}.")
            none = int(x.no_direct_timed + y.no_direct_timed)
            if none:
                parts.append(f"In {none} benchmarks on the {machine} {other} was timed.")
        if BEST_MODE_SENTENCE:
            ba, bt_ = get("best mode", "attention benchmark"), get("best mode", "training step benchmark")
            parts.append(f"Taking the faster timed mode on each side, the cache was faster in {ba.cache_faster} of {ba.comparable} "
                         f"attention benchmarks{_range(ba)}, and {bt_.cache_faster} of {bt_.comparable} training step "
                         f"benchmarks{_range(bt_)}.")
    excl = d[d.ceiling_exclusions > 0]
    if len(excl) == 0:
        parts.append("No direct method exceeded the ceiling.")
    else:
        parts.append(f"At least one direct method exceeded the ceiling in {excl.benchmark_id.nunique()} benchmarks, listed in "
                     f"Table~\\ref{{tab:ceiling}}.")
    rep = method_evaluations[method_evaluations.method.isin([CACHE_OF["replay"]] + list(DIRECT_OF["replay"]))]
    vs = rep.replay_verification.map(verification_status)
    n_vpass, n_vfail = int((vs == "passed").sum()), int((vs == "failed agreement").sum())
    n_rtimed = int((rep.status == "timed").sum())
    parts.append(f"Of {len(rep)} replay evaluations, {n_rtimed} were timed. Before timing, {n_vpass} passed replay verification "
                 f"and {n_vfail} failed it.")
    n_checks = len(agreement_checks)
    n_pass = int((agreement_checks.status == "passed").sum())
    n_fail = int((agreement_checks.status == "failed agreement").sum())
    n_other = n_checks - n_pass - n_fail
    ref = method_evaluations[method_evaluations.is_reference == True]
    n_ref, n_fin = len(ref), int((ref.reference_finite == True).sum())
    parts.append(f"Of {n_checks} agreement checks, {n_pass} passed, {n_fail} failed the tolerances and {n_other} did not run "
                 f"because the evaluation ended earlier. {n_fin} of {n_ref} reference evaluations were finite.")
    return " ".join(parts)

# ----- run on the combined datasets of cell 1, write the files -----
from apriori.serialization import save_to_pickle

NUMERIC = {"method_evaluations": ["execution_median_ms", "execution_min_ms", "execution_max_ms", "peak_gpu_allocated_median_mib",
                                  "peak_gpu_reserved_median_mib", "agreement_call_wall_ms", "warmups_completed", "timed_repeats_completed",
                                  "replay_setup_ms", "replay_capture_ms", "replay_reserved_after_capture_mib", "replay_mask_transfer_ms"],
           "preparation_measurements": ["wall_ms"],
           "benchmarks": ["n", "budget_M", "requested_run_length_lambda", "P", "coverage", "F", "N", "reuse_ratio",
                          "full_power_set_ratio", "D"]}
for name, cols in NUMERIC.items():
    df = combined[name]
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
benchmarks, method_evaluations, execution_calls, agreement_checks, preparation_measurements = (combined[k] for k in DATASETS)

derived = derive(benchmarks, method_evaluations, preparation_measurements)
best = best_modes(benchmarks, derived)
save_to_pickle(derived, os.path.join(TABLE_DIR, "derived.pkl.lz4"), lz4_compression=True)
save_to_pickle(best, os.path.join(TABLE_DIR, "derived_best_mode.pkl.lz4"), lz4_compression=True)

tables = {"tab_main_attention": table_main_attention(derived, METADATA),
          "tab_main_step": table_main_step(derived, METADATA),
          "tab_boundary": table_boundary(derived, METADATA),
          "tab_control": table_control(derived, METADATA),
          "tab_prep_attention": table_prep_attention(derived, METADATA),
          "tab_prep_step": table_prep_step(derived, METADATA),
          "tab_totals": table_totals(derived, METADATA),
          "tab_replay_setup": table_replay_setup(derived, METADATA),
          "tab_ceiling": table_ceiling(derived, METADATA)}
for name, tex in tables.items():
    path = os.path.join(TABLE_DIR, f"{name}.tex")
    if tex is None:
        if os.path.exists(path):
            os.remove(path)          # a stale table from an earlier run must not be pasted
        print(f"{name}: no rows, no file")
        continue
    with open(path, "w", encoding="utf-8") as f:
        f.write(tex)
    print(f"{name}: {len(tex):,} characters written")

para = paragraph(derived, best, agreement_checks, method_evaluations)
with open(os.path.join(TABLE_DIR, "paragraph.tex"), "w", encoding="utf-8") as f:
    f.write(para)
print("\n" + para + "\n")
win_counts(derived, best)

tab_main_attention: 3,189 characters written
tab_main_step: 2,461 characters written
tab_boundary: 2,129 characters written
tab_control: 2,234 characters written
tab_prep_attention: 2,864 characters written
tab_prep_step: 2,508 characters written
tab_totals: 2,495 characters written
tab_replay_setup: 2,305 characters written
tab_ceiling: 893 characters written

On the H100 the cache was faster than the fastest eager direct combination in 7 of 8 attention benchmarks, speedups $0.83$ to $3.49$, and 3 of 4 training step benchmarks, speedups $0.76$ to $2.84$. Cache replay was faster than the fastest masked attention replay in 8 of 8 attention benchmarks, speedups $2.06$ to $4.15$, and 4 of 4 training step benchmarks, speedups $1.64$ to $3.21$. Taking the faster timed mode on each side, the cache was faster in 8 of 8 attention benchmarks, speedups $2.06$ to $4.15$, and 4 of 4 training step benchmarks, speedups $1.64$ to $3.21$. On the 2080 the cache was faster than the fastest eager direct 

,pairing,machine,benchmark_type,experimental_role,benchmarks,comparable,cache_faster,min_speedup,max_speedup,cache_out_of_memory,missing,direct_out_of_memory,no_direct_timed,benchmarks_with_ceiling_exclusions,ceiling_exclusions
0,eager,2080,attention benchmark,fixed size run comparison,2,2,2,1.295215,1.888494,0,0,4.0,0.0,0.0,0.0
1,eager,2080,attention benchmark,main comparison,8,8,8,1.753363,3.903225,0,0,27.0,0.0,1.0,1.0
2,eager,2080,training step benchmark,main comparison,4,4,4,2.054197,3.022127,0,0,14.0,0.0,1.0,1.0
3,eager,H100,attention benchmark,fixed size run comparison,2,2,2,1.001467,1.596360,0,0,0.0,0.0,0.0,0.0
4,eager,H100,attention benchmark,main comparison,8,8,7,0.825036,3.489455,0,0,8.0,0.0,0.0,0.0
5,eager,H100,training step benchmark,main comparison,4,4,3,0.757004,2.838334,0,0,4.0,0.0,0.0,0.0
6,replay,2080,attention benchmark,fixed size run comparison,2,2,2,1.410637,2.058115,0,0,2.0,0.0,0.0,0.0
7,replay,2080,attention benchmark,main comparison,8,4,4,2.115955,4.731258,4,0,22.0,2.0,0.0,0.0
8,replay,2080,training step benchmark,main comparison,4,2,2,2.261492,3.003511,2,0,6.0,0.0,0.0,0.0
9,replay,H100,attention benchmark,fixed size run comparison,2,2,2,1.095136,1.709430,0,0,0.0,0.0,0.0,0.0


In [3]:
# Estimated run time: under 5 seconds. Preparation charges and preparation plus execution on the eager main comparison rows.
cols = ["machine", "benchmark_type", "n", "budget_M", "input_generation_median_ms", "cache_method_preparation_ms",
        "direct_method_preparation_ms", "comparator_direct_combination", "cache_execution_median_ms", "comparator_execution_median_ms",
        "cache_preparation_plus_execution_ms", "direct_preparation_plus_execution_ms", "preparation_plus_execution_ratio"]
prep_view = (derived[(derived["mode"] == "eager") & (derived.experimental_role == "main comparison")][cols]
             .sort_values(["benchmark_type", "n", "budget_M", "machine"]).reset_index(drop=True))
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
print(prep_view.to_string(index=False))
pd.reset_option("display.float_format")

machine          benchmark_type   n  budget_M  input_generation_median_ms  cache_method_preparation_ms  direct_method_preparation_ms comparator_direct_combination  cache_execution_median_ms  comparator_execution_median_ms  cache_preparation_plus_execution_ms  direct_preparation_plus_execution_ms  preparation_plus_execution_ratio
   2080     attention benchmark 300     64000                    2,651.23                        57.81                      2,828.27                     math@4096                      77.53                          177.44                               135.35                              3,005.71                             22.21
   H100     attention benchmark 300     64000                    3,078.30                        33.85                      1,344.00                  default@4096                      41.26                           34.04                                75.12                              1,378.04                             18.35
   2080

## Cell 3. Figure 1, speedup against budget

The code of Benchmark Tables and Figures v2, code cell 3, revised 2026-11-01 for the replay pairing, and writing the figure to `results\tables\fig1_speedup.png`. Four panels, attention and training step at n 300 and 496, speedup against requested budget M on a log axis, a dotted line at speedup 1. One series per machine and mode: color and marker shape give the machine, a solid line with filled markers the eager pairing (cache against the fastest eager direct combination), a dashed line with open markers the replay pairing (cache replay against the fastest masked attention replay). Every budget of the grid stays on the axis. Where a pairing has no speedup its line leaves a gap and the reason is marked at the baseline, a cross for the cache method out of memory, an open diamond for any other status. The cell ends by displaying the main comparison rows the figure draws from.

In [8]:
# Estimated run time: under 10 seconds. The code of Benchmark Tables and Figures v2, code cell 3, revised 2026-11-01 for the
# replay pairing.
# Figure 1. Speedup against requested budget M, four panels, from the derived main comparison rows. One series per machine
# and mode: solid for the eager pairing (cache against the fastest eager direct combination), dashed for the replay pairing
# (cache replay against the fastest masked attention replay). Every budget of the grid stays on the x axis. A benchmark without
# a speedup is left as a gap, so a line never crosses an unmeasured budget, and the gap is marked at the baseline with the
# reason: a cross for the cache method out of memory, an open diamond otherwise. The manifest fixes lambda = 10n.

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

FIG_PATH = os.path.join(TABLE_DIR, "fig1_speedup.png")
BUDGETS = {"attention benchmark": [64_000, 128_000, 256_000, 512_000], "training step benchmark": [64_000, 512_000]}
COLORS = {"2080": "#1f5fa8", "H100": "#c8551a"}
MARKERS = {"2080": "o", "H100": "s"}
LINES = {"eager": "-", "replay": "--"}
FILLS = {"eager": None, "replay": "white"}            # replay markers open, so mode never rests on line style alone
SERIES = [(machine, mode) for machine in ("2080", "H100") for mode in ("eager", "replay")]
JITTER = {s: 1.0 + j for s, j in zip(SERIES, (-0.045, -0.015, 0.015, 0.045))}
PANELS = [("attention benchmark", 300), ("attention benchmark", 496),
          ("training step benchmark", 300), ("training step benchmark", 496)]

def figure_speedup(derived, path=FIG_PATH, metadata=None):
    m = derived[derived.experimental_role == "main comparison"]
    fig, axes = plt.subplots(2, 2, figsize=(7.2, 5.6), sharex=True, sharey=True)
    ymin = 0.0
    sp = m.speedup.to_numpy(dtype=float)
    top = float(np.nanmax(sp)) if np.isfinite(sp).any() else 0.0   # an empty grid still draws the axes
    ymax = max(1.25, top * 1.08)                                   # the y = 1 reference line always shows
    for ax, (bt, n) in zip(axes.ravel(), PANELS):
        for machine, mode in SERIES:
            g = m[(m.benchmark_type == bt) & (m.n == n) & (m.machine == machine) & (m["mode"] == mode)
                  & (m.requested_run_length_lambda == 10 * n)]
            g = g.set_index("budget_M").reindex(BUDGETS[bt])   # every grid budget, NaN where no speedup
            y = g.speedup.to_numpy(dtype=float)
            x = np.array(BUDGETS[bt], dtype=float)
            color = COLORS[machine]
            ax.plot(x, y, color=color, marker=MARKERS[machine], ls=LINES[mode], lw=1.6, ms=5,
                    mfc=FILLS[mode] or color, label=f"{machine} {mode}")
            miss = ~np.isfinite(y)
            oom = miss & (g.cache_status.to_numpy() == "out of memory")
            other = miss & ~oom
            xj = x * JITTER[(machine, mode)]
            ax.plot(xj[oom], np.full(oom.sum(), ymin + 0.08), color=color, marker="x", ls="none", ms=5, mew=1.2)
            ax.plot(xj[other], np.full(other.sum(), ymin + 0.08), color=color, marker="D", ls="none", ms=4, mfc="none")
        ax.axhline(1.0, color="#777777", lw=1.0, ls=":")
        ax.set_xscale("log", base=2)
        ax.set_xticks(BUDGETS["attention benchmark"])
        ax.set_xticklabels(["64k", "128k", "256k", "512k"])
        ax.set_ylim(ymin, ymax)
        ax.set_title(f"{'Attention' if bt.startswith('attention') else 'Training step'}, $n = {n}$", fontsize=10)
        ax.grid(True, color="#e5e5e5", lw=0.6)
        ax.spines[["top", "right"]].set_visible(False)
    for ax in axes[1]:
        ax.set_xlabel("Requested budget $M$")
    for ax in axes[:, 0]:
        ax.set_ylabel("Speedup, direct / cache")
    h, l = axes[0, 0].get_legend_handles_labels()
    h += [plt.Line2D([], [], color="#555555", marker="x", ls="none", ms=5, mew=1.2),
          plt.Line2D([], [], color="#555555", marker="D", ls="none", ms=4, mfc="none")]
    l += ["cache method out of memory", "comparison unavailable, other status"]
    fig.legend(h, l, fontsize=7.5, frameon=False, loc="lower center", ncol=3)
    if metadata and metadata.get("mock", False):
        fig.text(0.5, 0.5, "MOCK VALUES, NOT MEASURED", ha="center", va="center", fontsize=28, color="#bbbbbb", alpha=0.35, rotation=20)
    fig.tight_layout(rect=(0, 0.1, 1, 1))
    fig.savefig(path, dpi=200)
    return path

print(figure_speedup(derived, metadata=METADATA))
derived[derived.experimental_role == "main comparison"][["machine", "benchmark_type", "n", "budget_M", "mode", "cache_status",
    "cache_execution_median_ms", "comparator_direct_combination", "comparator_execution_median_ms", "speedup"]].sort_values(
    ["benchmark_type", "n", "budget_M", "machine", "mode"]).reset_index(drop=True)

E:\apriori-ml-v3\results\tables\fig1_speedup.png


,machine,benchmark_type,n,budget_M,mode,cache_status,cache_execution_median_ms,comparator_direct_combination,comparator_execution_median_ms,speedup
0,2080,attention benchmark,300,64000,eager,timed,77.534600,math@4096,177.437499,2.288494
1,2080,attention benchmark,300,64000,replay,timed,74.018400,math@4096 replay,156.619600,2.115955
2,H100,attention benchmark,300,64000,eager,timed,41.264207,default@4096,34.044438,0.825036
3,H100,attention benchmark,300,64000,replay,timed,13.670764,default@all replay,28.142239,2.058571
4,2080,attention benchmark,300,128000,eager,timed,130.824900,math@4096,340.288400,2.601098
5,2080,attention benchmark,300,128000,replay,timed,94.312000,math@4096 replay,308.869600,3.274977
6,H100,attention benchmark,300,128000,eager,timed,49.878117,default@4096,64.582519,1.294807
7,H100,attention benchmark,300,128000,replay,timed,23.941812,default@all replay,55.896015,2.334661
8,2080,attention benchmark,300,256000,eager,timed,387.228199,math@4096,678.951500,1.753363
9,2080,attention benchmark,300,256000,replay,out of memory,NaN,math@4096 replay,624.841500,NaN
